# Behavior Classification: CNN + BiLSTM (v3 - Stability Fixed)

Pipeline yang telah diperbaiki:
- **Stabilitas**: Menggunakan `from_logits=True` dan menghapus `softmax` di layer terakhir untuk mencegah **Negative Loss**.
- **Multi-Class**: Mendukung 6 kategori perilaku (Layar, Materi, Tulis, Ponsel, Menoleh, Tidur).
- **Temporal**: Menggunakan `Bidirectional LSTM` untuk akurasi lebih tinggi.
- **Learning Rate**: Disesuaikan ke `5e-4` untuk konvergensi yang lebih mulus.

---

### Khusus Google Colab (Cloud)
Jika Anda menjalankan ini di Colab, jalankan sel di bawah ini untuk meng-clone project dari GitHub dan pindah ke direktori proyek.

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    repo_name = "skeleton-based-focus-estimation"
    repo_url = f"https://github.com/mdrnid/{repo_name}.git"
    
    if not os.path.exists(repo_name):
        print(f"Sedang meng-clone {repo_url}...")
        res = os.system(f"git clone {repo_url}")
        if res != 0:
            print("\n❌ ERROR: Gagal meng-clone repositori.")
        else:
            %cd {repo_name}
            print(f"✅ Berhasil masuk ke direktori: {os.getcwd()}")
    else:
        if not os.getcwd().endswith(repo_name):
            %cd {repo_name}
        print(f"✅ Direktori: {os.getcwd()}")

In [ ]:
import os
import sys
from pathlib import Path

def setup_paths():
    curr_dir = Path(os.getcwd())
    possible_roots = [
        curr_dir,
        curr_dir.parent,
        Path('/content/skeleton-based-focus-estimation'),
        Path('/content')
    ]
    for root in possible_roots:
        if (root / "data" / "processed").exists():
            print(f"✅ Root Proyek: {root.absolute()}")
            return str(root.absolute())
    print("⚠ WARNING: Folder 'data/processed' tidak ditemukan!")
    return '..'

BASE_PATH  = setup_paths()
DATA_PATH  = os.path.join(BASE_PATH, 'data', 'processed')
MODEL_PATH = os.path.join(BASE_PATH, 'models')

sys.path.insert(0, BASE_PATH)  # agar bisa import src.*
os.makedirs(MODEL_PATH, exist_ok=True)
print(f"Data Path : {DATA_PATH}")
print(f"Model Path: {MODEL_PATH}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, LSTM, Dense,
    Dropout, BatchNormalization, Input, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print(f"TensorFlow: {tf.__version__}")

## 1. Load Data (6 Subclasses)

In [ ]:
x_file = os.path.join(DATA_PATH, 'X.npy')
y_file = os.path.join(DATA_PATH, 'y.npy')

if os.path.exists(x_file):
    X = np.load(x_file)
    y = np.load(y_file)
    print(f"✅ X shape: {X.shape}  →  (Sampel, Timesteps=20, Fitur=70)")
    print(f"   y shape: {y.shape}")
    
    classes = np.unique(y)
    behavior_names = ["Layar", "Materi", "Tulis", "Ponsel", "Menoleh", "Tidur"]
    for cls in classes:
        count = (y == cls).sum()
        print(f"   [{cls}] {behavior_names[int(cls)]:13s}: {count} sampel")
    
    # Validasi shape
    assert X.shape[1] == 20, f"Timesteps harus 20, dapat {X.shape[1]}"
    assert X.shape[2] == 70, f"Fitur harus 70, dapat {X.shape[2]}"
    print("\n✅ Shape validasi: OK")
else:
    print(f"❌ ERROR: File data tidak ditemukan di {DATA_PATH}")
    print("   Jalankan dulu: python src/features/preprocess_csv.py")

## 2. Data Splitting (70/15/15)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Training  : {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test      : {X_test.shape}")

## 3. Class Weight

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, weights))
print(f"Class weights: {class_weight_dict}")

## 4. Bangun Model CNN + BiLSTM (STABLE)

In [ ]:
model = Sequential([
    Input(shape=(20, 70)),

    # --- Spatial Extraction ---
    Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),

    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.2),

    # --- Temporal Modeling ---
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.4),

    # --- Head ---
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(6)   # LOGITS: Tanpa Softmax biar stabil secara matematis
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4, clipnorm=1.0),
    # from_logits=True adalah KUNCI agar Loss tidak negatif lagi
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)
model.summary()

## 5. Training

In [ ]:
ckpt_path = os.path.join(MODEL_PATH, 'pose_model_best_v3.keras')

callbacks = [
    ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-6, verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

## 6. Evaluasi

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
y_pred_logits = model.predict(X_test)
y_pred = np.argmax(y_pred_logits, axis=1)

print("=" * 50)
print("Classification Report")
print("=" * 50)
behavior_names = ["Layar", "Materi", "Tulis", "Ponsel", "Menoleh", "Tidur"]
print(classification_report(
    y_test, y_pred,
    target_names=behavior_names
))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=behavior_names,
            yticklabels=behavior_names)
plt.title('Confusion Matrix — Test Set')
plt.show()

## 7. Backup ke Drive

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    drive_folder = '/content/drive/MyDrive/Models_Backup'
    os.makedirs(drive_folder, exist_ok=True)
    shutil.copy(ckpt_path, os.path.join(drive_folder, os.path.basename(ckpt_path)))
    print(f"✅ Model disimpan ke Drive: {drive_folder}")